In [1]:
import gzip
import math
import numpy as np
import random
import sklearn
import string
import pandas as pd
from collections import defaultdict
from gensim.models import Word2Vec
from nltk.stem.porter import *
from sklearn import linear_model
from sklearn.manifold import TSNE
import dateutil

In [2]:
import warnings
warnings.filterwarnings("ignore")

In [3]:
def assertFloat(x):
    assert type(float(x)) == float

def assertFloatList(items, N):
    assert len(items) == N
    assert [type(float(x)) for x in items] == [float]*N

In [21]:
data = []

f = gzip.open("renttherunway_final_data.json.gz")
for l in f:
    try:
        d = eval(l)
        data.append(d)
        if len(data) >= 150000:
            break
    except:
        None

f.close()

In [17]:
# Inputs: fit, weight, prior sizes, age, bodytype, bust size, category
# Outputs: size
# Model: 

In [5]:
allSizes = []
for d in data:
    allSizes.append([d['user_id'], d['size']])

userAverages = {}
sizesPerUser = defaultdict(list)
sizesTrain = allSizes[:75000]
sizesTest = allSizes[75000:]

for u, s in sizesTrain:
    sizesPerUser[u].append(s)

for u in sizesPerUser:
    us = [s for s in sizesPerUser[u]]
    userAverages[u] = sum(us) / len(us)

In [19]:
def parseFit(fit):
    return {'fit': 0,
           'small': -1,
           'large': 1}[fit]

def isInvalid(number):
    if number is None or pd.isnull(number):
        return True
    else:
        return False

letterSizes = {'aa' : 0,
              'a' : 1,
              'b' : 2,
              'c' : 3,
              'd' : 4,
              'e' : 5,
              'f' : 6,
              'g' : 7,
              'h' : 8,
              'i' : 9,
              'j' : 10,
              'k' : 11,
              'l' : 12,
              'm' : 13,
              'n' : 14,
              'o' : 15,
              'dd' : 5,
              'ddd' : 6,
              'dddd' : 7,
              'ddddd' : 8,
              'dddddd' : 9}

def parseBustSize(bustSize):
    if isInvalid(bustSize):
        return pd.Series([None, None])
    bandSize = re.search(r'[0-9]+', bustSize)[0]
    cupSize = letterSizes[re.search(r'[a-z]+', bustSize)[0]]
    return pd.Series([int(bandSize), cupSize])

def parseWeight(weight):
    if isInvalid(weight):
        return None
    return int(re.search(r'[0-9]+', weight)[0])

def parseHeight(height):
    try:
        height = re.sub("'", '', height)
        height = re.sub('"', '', height).split()
        return 12 * int(height[0]) + int(height[1])
    except:
        return None

In [22]:
df = pd.DataFrame.from_dict(data)
df

,fit,user_id,bust size,item_id,weight,rating,rented for,review_text,body type,review_summary,category,height,size,age,review_date
0,fit,420272,34d,2260466,137lbs,10,vacation,An adorable romper! Belt and zipper were a lit...,hourglass,So many compliments!,romper,"5' 8""",14,28,"April 20, 2016"
1,fit,273551,34b,153475,132lbs,10,other,I rented this dress for a photo shoot. The the...,straight & narrow,I felt so glamourous!!!,gown,"5' 6""",12,36,"June 18, 2013"
2,fit,360448,NaN,1063761,NaN,10,party,This hugged in all the right places! It was a ...,NaN,It was a great time to celebrate the (almost) ...,sheath,"5' 4""",4,116,"December 14, 2015"
3,fit,909926,34c,126335,135lbs,8,formal affair,I rented this for my company's black tie award...,pear,Dress arrived on time and in perfect condition.,dress,"5' 5""",8,34,"February 12, 2014"
4,fit,151944,34b,616682,145lbs,10,wedding,I have always been petite in my upper body and...,athletic,Was in love with this dress !!!,gown,"5' 9""",12,27,"September 26, 2016"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
149995,fit,14507,32a,1498329,124lbs,10,wedding,This dress fit true to size for the brand (whi...,pear,Perfect Formal Dress That Is Comfy!,gown,"5' 5""",1,29,"January 3, 2018"
149996,large,212893,34c,445639,112lbs,10,other,I loved this dress so much. It was perfect to ...,straight & narrow,Absolutely stunning dress,dress,"5' 5""",1,21,"June 4, 2017"
149997,small,920220,34d,1039951,128lbs,8,everyday,"Didn't work for me. I'm 5'5"", 34D, 128lbs. W...",full bust,"Very cute, but too small across chest and shor...",dress,"5' 5""",8,39,"October 27, 2017"
149998,fit,357454,36c,1980086,140lbs,10,formal affair,It is BEAUTIFUL and true to size. And although...,hourglass,I sparkled with class the entire evening!,gown,"5' 3""",8,47,"January 5, 2015"


In [26]:
df = pd.DataFrame.from_dict(data)
df['age'] = pd.to_numeric(df['age'])
df['fit'] = df['fit'].apply(parseFit)
df[['bust', 'cup']] = df['bust size'].apply(parseBustSize)
df['height'] = df['height'].apply(parseHeight)
df['weight'] = df['weight'].apply(parseWeight)
df = pd.concat([df, pd.get_dummies(df['body type'])], axis=1)
df = pd.concat([df, pd.get_dummies(df['category'])], axis=1)
df.drop(['category', 'item_id', 'rented for', 'body type', 'bust size', 'rating', 'review_text', 'review_summary', 'review_date'], axis=1, inplace=True)

In [8]:
df

,fit,user_id,weight,height,size,age,bust,cup,apple,athletic,...,tank,tee,tight,top,trench,trouser,trousers,tunic,turtleneck,vest
0,0,420272,137.0,68.0,14,28.0,34.0,4.0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,0,273551,132.0,66.0,12,36.0,34.0,2.0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,0,360448,NaN,64.0,4,116.0,NaN,NaN,0,0,...,0,0,0,0,0,0,0,0,0,0
3,0,909926,135.0,65.0,8,34.0,34.0,3.0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,0,151944,145.0,69.0,12,27.0,34.0,2.0,0,1,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
149995,0,14507,124.0,65.0,1,29.0,32.0,1.0,0,0,...,0,0,0,0,0,0,0,0,0,0
149996,1,212893,112.0,65.0,1,21.0,34.0,3.0,0,0,...,0,0,0,0,0,0,0,0,0,0
149997,-1,920220,128.0,65.0,8,39.0,34.0,4.0,0,0,...,0,0,0,0,0,0,0,0,0,0
149998,0,357454,140.0,63.0,8,47.0,36.0,3.0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [27]:
data = df.to_dict('records')

In [28]:
def feature(datum):
    feat = [1.0]
    for c in df.columns:
        if c not in ['size', 'user_id']:
            if pd.isna(datum[c]):
                feat.append(np.mean(df[c]))
            else:
                feat.append(datum[c])
    return feat

In [29]:
dataTrain = df[:75000]
dataTest = df[75000:]
cols = []
for col in dataTrain.columns:
    if col not in ['size', 'user_id']:
        cols.append(col)
Xtest = dataTest[cols]
ytest = dataTest['size'].reset_index(drop=True)
Xtrain = dataTrain[cols]
ytrain = dataTrain['size']


model = xgboost.XGBRegressor(objective='reg:squarederror', colsample_bytree = .5, max_depth = 10, alpha = 1)
model.fit(Xtrain, ytrain)
preds = model.predict(Xtest)
squares = []
for i in range(len(preds)):
    diffs = preds[i] - ytest[i]
    squares.append(diffs**2)
mse = sum(squares)/len(squares)

ValueError: Input contains NaN, infinity or a value too large for dtype('float64').

In [31]:
dataTrain = df[:75000]
dataTest = df[75000:]
Xtest = [feature(d) for d in dataTest]
ytest = [d['size'] for d in dataTest]
Xtrain = [feature(d) for d in dataTrain]
ytrain = [d['size'] for d in dataTrain]

TypeError: string indices must be integers

In [30]:
X = [feature(d) for d in data]
y = [d['size'] for d in data]

In [18]:
X, y

([[1.0,
   0,
   137.0,
   68.0,
   28.0,
   34.0,
   4.0,
   0,
   0,
   0,
   1,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   1,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0],
  [1.0,
   0,
   132.0,
   66.0,
   36.0,
   34.0,
   2.0,
   0,
   0,
   0,
   0,
   0,
   0,
   1,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   1,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,

In [23]:
model = linear_model.LinearRegression(fit_intercept=False)
model.fit(X, y)
y_pred = model.predict(X)
sse = sum([x**2 for x in (y - y_pred)])
mse = sse / len(y)
mse

26.359307083190593

In [ ]:
model = xgboost.XGBRegressor(objective='reg:squarederror', colsample_bytree = .5, max_depth = 10, alpha = 1)
model.fit(X, y)
preds = model.predict(Xtest)
MSE(preds, ytest)